# End-to-End Knot Geodesics with Plotly


In [1]:
!pip install plotly scikit-learn gudhi pygeodesic


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.8/6.8 MB 1.8 MB/s  0:00:03 eta 0:00:01
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.4
    Uninstalling numpy-1.26.4:
      Successfully uninstalled numpy-1.26.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
optax 0.2.5 requires jax>=0.4.27, which is not installed.
optax 0.2.5 requires jaxlib>=0.4.27, which is not installed.
orbax-checkpoint 0.11.5 requires jax>=0.4.34, which is not installed.
flax 0.10.4 requires jax>=0.4.27, which is not installed.
brax 0.12.5 requires jax>=0.4.6, which is not installed.
brax 0.12.5 requires jaxlib>=0.4.6, which is not installed.
brax 0.12.5 requires jaxopt, which is not installed.
chex 0.1.90 requires jax>=0.4.27, which is not installed.
chex 0.1.90 requires jaxlib>=0.4.27, which is not installed.
numba 0.61.2 requires numpy<2.3,>=1.24, but you have 

In [2]:
import numpy as np
import gudhi
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import dijkstra
import pygeodesic.geodesic as geodesic
import plotly.graph_objects as go
from sklearn.neighbors import NearestNeighbors
import warnings
warnings.filterwarnings('ignore')


### 1. Generate the Knot Manifold


In [3]:
def generate_tubular_knot_surface(n_points, p=3, q=2, r0=2.0, tube_r=0.2):
    np.random.seed(42)
    alpha = q / p
    t = np.random.rand(n_points) * 2*np.pi*p
    phi = np.random.rand(n_points) * 2*np.pi

    r = r0 + np.cos(alpha * t)
    dr = -alpha * np.sin(alpha * t)
    dz = alpha * np.cos(alpha * t)

    cos_t, sin_t = np.cos(t), np.sin(t)
    x_c, y_c, z_c = r * cos_t, r * sin_t, np.sin(alpha * t)
    tangents = np.stack([dr*cos_t - r*sin_t, dr*sin_t + r*cos_t, dz], axis=1)
    T = tangents / np.linalg.norm(tangents, axis=1, keepdims=True)

    a = np.tile([0,0,1], (n_points,1))
    mask = np.abs(np.sum(T * a, axis=1)) > 0.9
    a[mask] = [1,0,0]

    N = np.cross(a, T)
    N /= np.linalg.norm(N, axis=1, keepdims=True)
    B = np.cross(T, N)

    points = np.stack([x_c, y_c, z_c], axis=1) + tube_r * (np.cos(phi)[:,None]*N + np.sin(phi)[:,None]*B)
    return points


### 2. ISOMAP Calculation


In [4]:
def k_neighbors_graph(X, n_neighbors=8):
    nn = NearestNeighbors(n_neighbors=n_neighbors, algorithm='auto').fit(X)
    distances, indices = nn.kneighbors(X)
    n_points = X.shape[0]
    row, col, data = [], [], []
    for i in range(n_points):
        for j in range(1, n_neighbors):
            neighbor = indices[i, j]
            dist = distances[i, j]
            row.extend([i, neighbor])
            col.extend([neighbor, i])
            data.extend([dist, dist])
    return csr_matrix((data, (row, col)), shape=(n_points, n_points))

def get_shortest_path(predecessors, start_idx, end_idx):
    path = []
    curr = end_idx
    while curr != -9999 and curr >= 0:
        path.append(curr)
        if curr == start_idx:
            break
        curr = predecessors[start_idx, curr]
    return path[::-1]


### 3. TDC Mesh Generation & 2-Manifold Pruning


In [5]:
def reconstruct_surface_tdc(points, max_edge_length_squared=0.2):
    print("Computing Tangential Complex...")

    np.random.seed(42)
    jitter = np.random.normal(scale=1e-5, size=points.shape)
    perturbed_points = points + jitter

    tc = gudhi.TangentialComplex(intrisic_dim=2, points=perturbed_points)
    if max_edge_length_squared is not None:
        tc.set_max_squared_edge_length(max_edge_length_squared)
    tc.compute_tangential_complex()

    st = tc.create_simplex_tree()

    raw_triangles = []
    for simplex, _ in st.get_simplices():
        if len(simplex) == 3:
            raw_triangles.append(simplex)

    print(f"Extracted {len(raw_triangles)} raw triangles from the complex.")

    edge_counts = {}
    for t in raw_triangles:
        for i in range(3):
            e = tuple(sorted((t[i], t[(i+1)%3])))
            edge_counts[e] = edge_counts.get(e, 0) + 1

    current_triangles = list(raw_triangles)
    dropped = 0

    while True:
        non_manifold_edges = {e: c for e, c in edge_counts.items() if c > 2}
        if not non_manifold_edges:
            break

        worst_edge = max(non_manifold_edges, key=non_manifold_edges.get)
        bad_tris_indices = []
        for idx, t in enumerate(current_triangles):
            for i in range(3):
                e = tuple(sorted((t[i], t[(i+1)%3])))
                if e == worst_edge:
                    bad_tris_indices.append(idx)
                    break

        drop_idx = bad_tris_indices[-1]
        drop_t = current_triangles.pop(drop_idx)

        for i in range(3):
            e = tuple(sorted((drop_t[i], drop_t[(i+1)%3])))
            edge_counts[e] -= 1
            if edge_counts[e] == 0:
                del edge_counts[e]

        dropped += 1

    if dropped > 0:
        print(f"Pruned exactly {dropped} overlapping triangles to enforce strict 2-manifold geometry.")

    return np.array(current_triangles)

def compute_tdc_distances(points, triangles):
    n_points = len(points)
    if len(triangles) == 0:
        return np.full((n_points, n_points), np.inf), None

    geoalg = geodesic.PyGeodesicAlgorithmExact(points, triangles)
    dist_matrix = np.zeros((n_points, n_points))

    print("Computing exact surface geodesics...")
    for i in range(n_points):
        distances, _ = geoalg.geodesicDistances(np.array([i]))
        dist_matrix[i, :] = distances

    disconnected_mask = dist_matrix > 1e15
    if disconnected_mask.any():
        max_dist = dist_matrix[~disconnected_mask].max()
        if max_dist == 0:
            max_dist = 1.0
        dist_matrix[disconnected_mask] = max_dist * 10

    return dist_matrix, geoalg


### 4. Execute the Pipeline\nWe use a denser sampling of $N=2500$ points here so the Knot holds together nicely when enforcing the metric limitation and pruning overlapping faces.


In [6]:
# Constants
N_POINTS = 5000
START_IDX = 0
END_IDX = N_POINTS // 2
K_ISOMAP = 10
MAX_EDGE = 0.3

p=4
q=5
r0=2
tube_r=0.2

# 1. Generate points
X = generate_tubular_knot_surface(N_POINTS, p, q, r0, tube_r)
print(f"Dataset generated. {len(X)} points.")

# 2. ISOMAP
iso_graph = k_neighbors_graph(X, n_neighbors=K_ISOMAP)
iso_dist_matrix, iso_predecessors = dijkstra(csgraph=iso_graph, directed=False, return_predecessors=True)
iso_dist = iso_dist_matrix[START_IDX, END_IDX]
iso_path_indices = get_shortest_path(iso_predecessors, START_IDX, END_IDX)
iso_path_points = X[iso_path_indices] if len(iso_path_indices) > 0 else []
print(f"ISOMAP Estimated Distance: {iso_dist:.4f}")

# 3. TDC
triangles = reconstruct_surface_tdc(X, max_edge_length_squared=MAX_EDGE)
tdc_dist_matrix, geoalg = compute_tdc_distances(X, triangles)
if geoalg is not None:
    tdc_dist = tdc_dist_matrix[START_IDX, END_IDX]
    _, path_points = geoalg.geodesicDistance(START_IDX, END_IDX)
    tdc_path_points = np.array(path_points)
    print(f"TDC Estimated Distance: {tdc_dist:.4f}")
else:
    tdc_dist = np.inf
    tdc_path_points = []



Dataset generated. 5000 points.
ISOMAP Estimated Distance: 34.5278
Computing Tangential Complex...
Extracted 10849 raw triangles from the complex.
Pruned exactly 1105 overlapping triangles to enforce strict 2-manifold geometry.
Computing exact surface geodesics...
TDC Estimated Distance: 26.5939


### 5. Interactive 3D Visualization using Plotly


In [ ]:
fig = go.Figure()

# Plot the base point cloud (semi-transparent)
fig.add_trace(go.Scatter3d(
    x=X[:, 0], y=X[:, 1], z=X[:, 2],
    mode='markers',
    marker=dict(size=2, color='lightblue', opacity=0.3),
    name='Point Cloud'
))

# Plot the Mesh
if len(triangles) > 0:
    fig.add_trace(go.Mesh3d(
        x=X[:,0], y=X[:,1], z=X[:,2],
        i=triangles[:,0], j=triangles[:,1], k=triangles[:,2],
        color='lightgreen',
        opacity=0.3,
        name='TDC Mesh',
        hoverinfo='skip'
    ))

# Plot ISOMAP path
if len(iso_path_points) > 0:
    fig.add_trace(go.Scatter3d(
        x=iso_path_points[:, 0],
        y=iso_path_points[:, 1],
        z=iso_path_points[:, 2],
        mode='lines+markers',
        line=dict(color='blue', width=6),
        marker=dict(size=4, color='blue'),
        name=f'ISOMAP Path (k={K_ISOMAP})'
    ))

# Plot TDC Path
if len(tdc_path_points) > 0:
    fig.add_trace(go.Scatter3d(
        x=tdc_path_points[:, 0],
        y=tdc_path_points[:, 1],
        z=tdc_path_points[:, 2],
        mode='lines+markers',
        line=dict(color='magenta', width=8),
        marker=dict(size=4, color='magenta'),
        name='TDC Exact Path'
    ))

# Start and End markers
fig.add_trace(go.Scatter3d(
    x=[X[START_IDX, 0]], y=[X[START_IDX, 1]], z=[X[START_IDX, 2]],
    mode='markers',
    marker=dict(size=8, color='green', symbol='circle'),
    name='Start'
))
fig.add_trace(go.Scatter3d(
    x=[X[END_IDX, 0]], y=[X[END_IDX, 1]], z=[X[END_IDX, 2]],
    mode='markers',
    marker=dict(size=8, color='red', symbol='circle'),
    name='End'
))

fig.update_layout(
    title=f"Knot Geodesics: ISOMAP vs Exact TDC<br><sup>ISOMAP: {iso_dist:.4f}  |  TDC: {tdc_dist:.4f}</sup>",
    scene=dict(
        xaxis=dict(visible=False),
        yaxis=dict(visible=False),
        zaxis=dict(visible=False),
        aspectmode='data'
    ),
    margin=dict(l=0, r=0, b=0, t=40),
    legend=dict(yanchor="top", y=0.9, xanchor="left", x=0.1)
)

fig.show()